In [ ]:
import os
import sys
from pathlib import Path

BASE_DIR = Path("/content/helpdesk-llm-rag-implementos")
SRC_DIR = BASE_DIR / "src"

BASE_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

# crea __init__.py si no existe
(BASE_DIR / "__init__.py").write_text("", encoding="utf-8")
(SRC_DIR / "__init__.py").write_text("", encoding="utf-8")

# agrega la carpeta raíz al path
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print("BASE_DIR existe:", BASE_DIR.exists())
print("SRC_DIR existe:", SRC_DIR.exists())
print("En sys.path:", str(BASE_DIR) in sys.path)
print("sys.path[0]:", sys.path[0])

BASE_DIR existe: True
SRC_DIR existe: True
En sys.path: True
sys.path[0]: /content


In [ ]:
from pathlib import Path

codigo = r'''from __future__ import annotations

from typing import Dict, List
from src.classifier import classify_query
from src.retriever import retrieve, RetrievedDocument
from src.validator import assess_confidence, should_escalate


class HelpdeskRAGPipeline:
    def _build_answer(
        self,
        query: str,
        docs: List[RetrievedDocument],
        confidence: str,
        escalate: bool
    ) -> str:
        if escalate or not docs:
            return (
                "No se encontró evidencia documental suficiente para entregar una respuesta confiable. "
                "Se recomienda derivar el caso a un analista humano para revisión."
            )

        evidence_lines = []
        for doc in docs:
            snippet = doc.content.split(".")[0].strip()
            evidence_lines.append(f"- ({doc.origin}) {snippet}.")

        return (
            f"Para la consulta '{query}', el sistema recuperó evidencia relevante y estima una confianza {confidence}. "
            "Con base en los documentos encontrados, se sugiere la siguiente orientación:\n"
            + "\n".join(evidence_lines)
            + "\n\nRespuesta sintetizada: siga el procedimiento indicado en los documentos fuente y, "
            + "si el problema persiste, contacte a soporte de segundo nivel."
        )

    def process_query(self, query: str) -> Dict[str, object]:
        category = classify_query(query)
        docs = retrieve(query)
        confidence = assess_confidence(docs)
        escalate = should_escalate(confidence)
        answer = self._build_answer(query, docs, confidence, escalate)

        return {
            "category": category,
            "confidence": confidence,
            "escalate": escalate,
            "answer": answer,
            "sources": [f"{doc.source} ({doc.origin})" for doc in docs] if docs else ["Sin fuentes suficientes"],
        }
'''

Path("/content/helpdesk-llm-rag-implementos/src/rag_pipeline.py").write_text(codigo, encoding="utf-8")
print("rag_pipeline.py creado correctamente")

rag_pipeline.py creado correctamente


In [ ]:
from pathlib import Path
import json
import textwrap

BASE_DIR = Path('/content/helpdesk-llm-rag-implementos')
SRC_DIR = BASE_DIR / 'src'
DATA_INTERNAL = BASE_DIR / 'data' / 'internal'
DATA_EXTERNAL = BASE_DIR / 'data' / 'external'
TESTS_DIR = BASE_DIR / 'tests'
DOCS_DIR = BASE_DIR / 'docs'
EVIDENCIAS_DIR = BASE_DIR / 'evidencias'

for d in [SRC_DIR, DATA_INTERNAL, DATA_EXTERNAL, TESTS_DIR, DOCS_DIR, EVIDENCIAS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

files = {
    BASE_DIR / 'app.py': '''from src.rag_pipeline import HelpdeskRAGPipeline

def main() -> None:
    pipeline = HelpdeskRAGPipeline()
    print("=== Helpdesk con LLM + RAG (prototipo académico) ===")
    print("Escribe tu consulta. Para salir, escribe 'salir'.\\n")

    while True:
        query = input("Consulta: ").strip()
        if query.lower() in {"salir", "exit", "quit"}:
            print("Sesión finalizada.")
            break

        result = pipeline.process_query(query)

        print("\\n--- Resultado ---")
        print(f"Categoría: {result['category']}")
        print(f"Confianza: {result['confidence']}")
        print(f"Escalar: {'Sí' if result['escalate'] else 'No'}")
        print("\\nRespuesta:")
        print(result["answer"])

        print("\\nFuentes:")
        for source in result["sources"]:
            print(f"- {source}")

        print("\\n" + "=" * 50 + "\\n")

if __name__ == "__main__":
    main()
''',
    SRC_DIR / '__init__.py': '''''',
    SRC_DIR / 'classifier.py': '''from __future__ import annotations

def classify_query(query: str) -> str:
    q = query.lower()

    if any(word in q for word in ["contraseña", "clave", "password", "autenticación", "login", "ingresar"]):
        return "Incidente técnico"
    if any(word in q for word in ["vpn", "instalar", "configurar", "acceso", "habilitar"]):
        return "Requerimiento de servicio"
    if any(word in q for word in ["horario", "atención", "soporte", "contacto", "correo"]):
        return "Pregunta frecuente"
    if any(word in q for word in ["permiso", "solicitud", "administrativo"]):
        return "Consulta administrativa"

    return "Caso no clasificable"
''',
    SRC_DIR / 'prompts.py': '''SYSTEM_PROMPT = """
Eres un asistente de helpdesk de Implementos.
Responde solo con base en la evidencia recuperada.
Si la evidencia es insuficiente, indica que el caso debe ser revisado por un analista humano.
Incluye fuentes y nivel de confianza.
""".strip()

CLASSIFICATION_PROMPT = """
Clasifica la consulta del usuario en una de estas categorías:
- Incidente técnico
- Requerimiento de servicio
- Consulta administrativa
- Pregunta frecuente
- Caso no clasificable
""".strip()

ESCALATION_PROMPT = """
Resume el caso para derivarlo a un analista humano.
Incluye problema, evidencia encontrada, motivo de escalamiento y prioridad sugerida.
""".strip()
''',
    SRC_DIR / 'retriever.py': '''from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
from typing import List

BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data"

STOPWORDS = {
    "para", "con", "sin", "una", "uno", "del", "las", "los", "que", "por", "este",
    "esta", "como", "donde", "cuando", "sobre", "debe", "deben", "de", "la", "el",
    "mi", "mis", "sus", "hay", "más", "muy", "necesito", "ayuda", "caso", "sistema",
    "usuario", "equipo", "soporte", "proceso", "documentación", "modulo", "módulo",
}

@dataclass
class RetrievedDocument:
    source: str
    origin: str
    score: int
    content: str

def tokenize(text: str) -> set[str]:
    tokens = set(re.findall(r"[a-záéíóúñA-ZÁÉÍÓÚÑ]{3,}", text.lower()))
    return {t for t in tokens if t not in STOPWORDS}

def load_documents() -> list[tuple[str, str, str]]:
    documents: list[tuple[str, str, str]] = []
    for origin in ("internal", "external"):
        folder = DATA_DIR / origin
        for path in sorted(folder.glob("*.txt")):
            documents.append((path.name, origin, path.read_text(encoding="utf-8")))
    return documents

def retrieve(query: str, top_k: int = 3) -> List[RetrievedDocument]:
    query_tokens = tokenize(query)
    retrieved: list[RetrievedDocument] = []

    for filename, origin, content in load_documents():
        content_tokens = tokenize(content)
        score = len(query_tokens.intersection(content_tokens))
        if score > 0:
            retrieved.append(
                RetrievedDocument(
                    source=filename,
                    origin=origin,
                    score=score,
                    content=content.strip(),
                )
            )

    retrieved.sort(key=lambda doc: doc.score, reverse=True)
    return retrieved[:top_k]
''',
    SRC_DIR / 'validator.py': '''from __future__ import annotations

from typing import List
from src.retriever import RetrievedDocument

def assess_confidence(retrieved_docs: List[RetrievedDocument]) -> str:
    if not retrieved_docs:
        return "baja"

    top_score = retrieved_docs[0].score

    if top_score >= 2:
        return "alta"
    if top_score >= 1:
        return "media"
    return "baja"

def should_escalate(confidence: str) -> bool:
    return confidence == "baja"
''',
    SRC_DIR / 'rag_pipeline.py': '''from __future__ import annotations

from typing import Dict, List
from src.classifier import classify_query
from src.retriever import retrieve, RetrievedDocument
from src.validator import assess_confidence, should_escalate

class HelpdeskRAGPipeline:
    def _build_answer(self, query: str, docs: List[RetrievedDocument], confidence: str, escalate: bool) -> str:
        if escalate or not docs:
            return (
                "No se encontró evidencia documental suficiente para entregar una respuesta confiable. "
                "Se recomienda derivar el caso a un analista humano para revisión."
            )

        evidence_lines = []
        for doc in docs:
            snippet = doc.content.split(".")[0].strip()
            evidence_lines.append(f"- ({doc.origin}) {snippet}.")

        return (
            f"Para la consulta '{query}', el sistema recuperó evidencia relevante y estima una confianza {confidence}. "
            "Con base en los documentos encontrados, se sugiere la siguiente orientación:\\n"
            + "\\n".join(evidence_lines)
            + "\n\nRespuesta sintetizada: siga el procedimiento indicado en los documentos fuente y, "
              "si el problema persiste, contacte a soporte de segundo nivel."
        )

    def process_query(self, query: str) -> Dict[str, object]:
        category = classify_query(query)
        docs = retrieve(query)
        confidence = assess_confidence(docs)
        escalate = should_escalate(confidence)
        answer = self._build_answer(query, docs, confidence, escalate)

        return {
            "category": category,
            "confidence": confidence,
            "escalate": escalate,
            "answer": answer,
            "sources": [f"{doc.source} ({doc.origin})" for doc in docs] if docs else ["Sin fuentes suficientes"],
        }
''',
    DATA_INTERNAL / 'faq_soporte.txt': '''Preguntas frecuentes de soporte.
El horario de atención de mesa de ayuda es de lunes a viernes de 08:30 a 18:00.
Las solicitudes urgentes deben registrarse en la plataforma de tickets y marcarse con prioridad alta.
''',
    DATA_INTERNAL / 'manual_restablecimiento_contrasena.txt': '''Procedimiento interno para restablecimiento de contraseña.
El usuario debe ingresar al portal de soporte, seleccionar la opción 'Olvidé mi contraseña' y validar su identidad con el correo corporativo.
Si el error persiste después del segundo intento, el caso debe ser derivado al soporte de segundo nivel.
''',
    DATA_INTERNAL / 'politica_autenticacion.txt': '''Política interna de autenticación.
Toda cuenta corporativa requiere una contraseña segura, verificación por correo y cambio obligatorio cada 90 días.
Los bloqueos por intentos fallidos deben ser revisados por el equipo de soporte.
''',
    DATA_EXTERNAL / 'guia_buenas_practicas_credenciales.txt': '''Guía externa de buenas prácticas para gestión de credenciales.
Se recomienda habilitar procesos de recuperación con validación por correo, autenticación multifactor y registro de auditoría para cambios de contraseña.
''',
    DATA_EXTERNAL / 'recomendaciones_mesa_ayuda.txt': '''Buenas prácticas de mesa de ayuda.
Los sistemas de soporte deben mostrar fuentes, registrar trazabilidad y escalar consultas ambiguas o de baja confianza a un analista humano.
''',
    DATA_EXTERNAL / 'referencia_vpn_trabajo_remoto.txt': '''Referencia técnica externa sobre VPN corporativa.
La instalación de VPN debe realizarse con cliente autorizado, credenciales corporativas y validación del área TI antes del primer acceso remoto.
''',
    TESTS_DIR / 'casos_prueba.json': json.dumps([
        {
            "query": "¿Cómo restablezco mi contraseña del sistema?",
            "expected_category": "Incidente técnico",
            "expected_confidence": "media"
        },
        {
            "query": "¿Cuál es el horario de atención del soporte?",
            "expected_category": "Pregunta frecuente"
        },
        {
            "query": "Necesito instalar VPN en mi equipo corporativo",
            "expected_category": "Requerimiento de servicio"
        },
        {
            "query": "Necesito ayuda con sincronización cuántica holográfica del módulo ZX-99",
            "expected_escalate": True
        }
    ], ensure_ascii=False, indent=2),
    TESTS_DIR / 'test_pipeline.py': '''import json
from pathlib import Path
import unittest

from src.rag_pipeline import HelpdeskRAGPipeline

class TestHelpdeskRAGPipeline(unittest.TestCase):
    def setUp(self) -> None:
        self.pipeline = HelpdeskRAGPipeline()

    def test_cases_from_json(self) -> None:
        path = Path(__file__).parent / "casos_prueba.json"
        cases = json.loads(path.read_text(encoding="utf-8"))

        for case in cases:
            result = self.pipeline.process_query(case["query"])

            if "expected_category" in case:
                self.assertEqual(result["category"], case["expected_category"])
            if "expected_confidence" in case:
                self.assertEqual(result["confidence"], case["expected_confidence"])
            if "expected_escalate" in case:
                self.assertEqual(result["escalate"], case["expected_escalate"])

if __name__ == "__main__":
    unittest.main()
''',
}

for path, content in files.items():
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

print('Proyecto creado en:', BASE_DIR)
for path in sorted(BASE_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(BASE_DIR))


Proyecto creado en: /content/helpdesk-llm-rag-implementos
app.py
data/external/guia_buenas_practicas_credenciales.txt
data/external/recomendaciones_mesa_ayuda.txt
data/external/referencia_vpn_trabajo_remoto.txt
data/internal/faq_soporte.txt
data/internal/manual_restablecimiento_contrasena.txt
data/internal/politica_autenticacion.txt
src/__init__.py
src/classifier.py
src/prompts.py
src/rag_pipeline.py
src/retriever.py
src/validator.py
tests/casos_prueba.json
tests/test_pipeline.py


## Probar una consulta

In [ ]:
import sys
sys.path.append(str(BASE_DIR))

from src.rag_pipeline import HelpdeskRAGPipeline

pipeline = HelpdeskRAGPipeline()
consulta = '¿Cómo restablezco mi contraseña del sistema?'
resultado = pipeline.process_query(consulta)
resultado


{'category': 'Incidente técnico',
 'confidence': 'media',
 'escalate': False,
 'answer': "Para la consulta '¿Cómo restablezco mi contraseña del sistema?', el sistema recuperó evidencia relevante y estima una confianza media. Con base en los documentos encontrados, se sugiere la siguiente orientación:\n- (internal) Procedimiento interno para restablecimiento de contraseña.\n- (internal) Política interna de autenticación.\n- (external) Guía externa de buenas prácticas para gestión de credenciales.\n\nRespuesta sintetizada: siga el procedimiento indicado en los documentos fuente y, si el problema persiste, contacte a soporte de segundo nivel.",
 'sources': ['manual_restablecimiento_contrasena.txt (internal)',
  'politica_autenticacion.txt (internal)',
  'guia_buenas_practicas_credenciales.txt (external)']}

## Probar varias consultas manualmente

In [ ]:
consultas = [
    '¿Cómo restablezco mi contraseña del sistema?',
    '¿Cuál es el horario de atención del soporte?',
    'Necesito instalar VPN en mi equipo corporativo',
    'Necesito ayuda con sincronización cuántica holográfica del módulo ZX-99'
]

for consulta in consultas:
    print('=' * 80)
    print('Consulta:', consulta)
    print(pipeline.process_query(consulta))
    print()


Consulta: ¿Cómo restablezco mi contraseña del sistema?
{'category': 'Incidente técnico', 'confidence': 'media', 'escalate': False, 'answer': "Para la consulta '¿Cómo restablezco mi contraseña del sistema?', el sistema recuperó evidencia relevante y estima una confianza media. Con base en los documentos encontrados, se sugiere la siguiente orientación:\n- (internal) Procedimiento interno para restablecimiento de contraseña.\n- (internal) Política interna de autenticación.\n- (external) Guía externa de buenas prácticas para gestión de credenciales.\n\nRespuesta sintetizada: siga el procedimiento indicado en los documentos fuente y, si el problema persiste, contacte a soporte de segundo nivel.", 'sources': ['manual_restablecimiento_contrasena.txt (internal)', 'politica_autenticacion.txt (internal)', 'guia_buenas_practicas_credenciales.txt (external)']}

Consulta: ¿Cuál es el horario de atención del soporte?
{'category': 'Pregunta frecuente', 'confidence': 'alta', 'escalate': False, 'answe

## Modo interactivo

In [ ]:

%cd /content/helpdesk-llm-rag-implementos
!python app.py


/content/helpdesk-llm-rag-implementos
=== Helpdesk con LLM + RAG (prototipo académico) ===
Escribe tu consulta. Para salir, escribe 'salir'.

Consulta: horario de atencion 

--- Resultado ---
Categoría: Pregunta frecuente
Confianza: media
Escalar: No

Respuesta:
Para la consulta 'horario de atencion', el sistema recuperó evidencia relevante y estima una confianza media. Con base en los documentos encontrados, se sugiere la siguiente orientación:
- (internal) Preguntas frecuentes de soporte.

Respuesta sintetizada: siga el procedimiento indicado en los documentos fuente y, si el problema persiste, contacte a soporte de segundo nivel.

Fuentes:
- faq_soporte.txt (internal)


Consulta: Traceback (most recent call last):
  File "/content/helpdesk-llm-rag-implementos/app.py", line 30, in <module>
    main()
  File "/content/helpdesk-llm-rag-implementos/app.py", line 9, in main
    query = input("Consulta: ").strip()
            ^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
^C
